# Challenge 01 — VQE: Ground-State Energy of the H₂ Molecule
**Qiskit Fall Fest 2026 · Team Cdevi Code · Intermediate**

## Goal
Find the **lowest possible energy** (the *ground-state energy*) of a hydrogen molecule, H₂, using the
**Variational Quantum Eigensolver (VQE)**, and prove the answer is right by comparing it with an
exact classical calculation.

## Why does this matter?
The ground-state energy of a molecule decides its shape, how strongly its atoms are bound and how it
reacts. Computing it exactly on a classical computer gets exponentially expensive as molecules grow.
Quantum computers may handle this better because a register of qubits can naturally represent the
quantum state of electrons.

## Roadmap of this notebook
| Phase | What we do | Output |
|---|---|---|
| **1** | Define H₂ in PySCF → build its Hamiltonian → map to qubits → **exact classical answer** | the reference energy |
| 2 | Build the ansatz (parameterised quantum circuit) | a circuit with tunable angles |
| 3 | Run VQE (quantum circuit + classical optimizer) | VQE energy |
| 4 | Compare VQE vs exact, analyse error, discuss | verification and plots |

> **Units:** energies are in **Hartree (Ha)**, the natural unit of quantum chemistry
> (1 Ha ≈ 27.211 eV ≈ 627.5 kcal/mol). *Chemical accuracy* is a target error of
> **1.6 mHa** (= 1 kcal/mol); below it, a computed energy is accurate enough to predict chemistry.

## 0. Setup: imports and versions
Qiskit's API changes between releases, so we **print every library version**. Anyone re-running this
notebook can then check they have the same environment (see `requirements.txt`).

| Library | Role in this notebook |
|---|---|
| `pyscf` | Classical quantum-chemistry engine: computes the integrals that define the H₂ Hamiltonian |
| `qiskit_nature` | Bridges chemistry and Qiskit: turns PySCF output into a fermionic Hamiltonian and maps it onto qubits |
| `qiskit` | Core quantum-circuit and operator library (`SparsePauliOp`) |
| `qiskit_algorithms` | Ready-made algorithms: `NumPyMinimumEigensolver` (exact) and later `VQE` |
| `numpy` | Linear algebra for our independent cross-check |

In [1]:
import numpy as np

import pyscf
import qiskit
import qiskit_nature
import qiskit_algorithms

# Chemistry -> Qiskit bridge
from qiskit_nature.units import DistanceUnit                         # lets us say "Angstrom" explicitly
from qiskit_nature.second_q.drivers import PySCFDriver               # runs PySCF for us
from qiskit_nature.second_q.mappers import JordanWignerMapper        # fermions -> qubits
from qiskit_nature.second_q.algorithms import GroundStateEigensolver # chemistry-aware solver wrapper

# Exact (classical) eigensolver used as our ground truth
from qiskit_algorithms import NumPyMinimumEigensolver

# Independent cross-check with PySCF's own Full Configuration Interaction (FCI) solver
from pyscf import gto, scf, fci

print(f"qiskit            {qiskit.__version__}")
print(f"qiskit-nature     {qiskit_nature.__version__}")
print(f"qiskit-algorithms {qiskit_algorithms.__version__}")
print(f"pyscf             {pyscf.__version__}")
print(f"numpy             {np.__version__}")

qiskit            2.2.3
qiskit-nature     0.8.0
qiskit-algorithms 0.4.0
pyscf             2.15.0
numpy             2.2.6


## 1. Define the H₂ molecule with PySCF

To describe a molecule we must choose four things:

1. **Geometry**: where the nuclei sit. H₂ has two hydrogen nuclei on the z-axis, **0.735 Å** apart.
   That is close to H₂'s experimental equilibrium bond length (≈ 0.741 Å), the distance where the
   molecule is most stable.
2. **Charge = 0**: a neutral molecule, so 2 protons and **2 electrons**.
3. **Spin = 0**: PySCF's `spin` is 2S, the number of unpaired electrons. Both electrons are paired
   (one spin-up ↑, one spin-down ↓), which is the *singlet* ground state.
4. **Basis set = STO-3G**: electrons live in *orbitals*, and a basis set is the menu of mathematical
   functions used to build them. STO-3G is the smallest standard basis: **one 1s function per H atom**.
   Two atoms give 2 *spatial orbitals*, which combine into
   - a **bonding** orbital σg (lower energy, electrons pulled between the nuclei), and
   - an **antibonding** orbital σu* (higher energy).

**Behind the scenes, the `PySCFDriver`:**
- runs a **Hartree–Fock (HF)** calculation, a mean-field approximation in which each electron feels
  only the *average* repulsion of the others;
- uses the HF molecular orbitals to compute the **one- and two-electron integrals**. These numbers are
  exactly what the Hamiltonian needs.

HF misses *electron correlation* (electrons actively avoiding each other). The gap between the HF
energy and the exact energy is the **correlation energy**, and recovering it is the whole point of VQE.

In [2]:
# --- Molecule specification -------------------------------------------------
BOND_LENGTH = 0.735  # Angstrom, close to the H2 equilibrium bond length

driver = PySCFDriver(
    atom=f"H 0 0 0; H 0 0 {BOND_LENGTH}",  # "element x y z; element x y z"
    basis="sto3g",                          # minimal basis: one 1s orbital per H atom
    charge=0,                               # neutral molecule -> 2 electrons
    spin=0,                                 # 2S = 0 -> all electrons paired (singlet)
    unit=DistanceUnit.ANGSTROM,             # coordinates above are in Angstrom
)

# driver.run() calls PySCF (Hartree-Fock + integrals) and packages everything
# into an ElectronicStructureProblem object.
problem = driver.run()

n_spatial = problem.num_spatial_orbitals   # number of molecular orbitals
n_alpha, n_beta = problem.num_particles    # (# spin-up electrons, # spin-down electrons)
nuclear_repulsion = problem.hamiltonian.nuclear_repulsion_energy
hf_energy = problem.reference_energy       # Hartree-Fock total energy

print(f"Spatial orbitals          : {n_spatial}")
print(f"Spin orbitals (= qubits)  : {2 * n_spatial}")
print(f"Electrons (alpha, beta)   : ({n_alpha}, {n_beta})")
print(f"Nuclear repulsion energy  : {nuclear_repulsion:.6f} Ha")
print(f"Hartree-Fock total energy : {hf_energy:.6f} Ha")

Spatial orbitals          : 2
Spin orbitals (= qubits)  : 4
Electrons (alpha, beta)   : (1, 1)
Nuclear repulsion energy  : 0.719969 Ha
Hartree-Fock total energy : -1.116999 Ha


**Reading the output**
- **2 spatial orbitals × 2 spin directions = 4 spin orbitals**, and each spin orbital will become one qubit.
- **Nuclear repulsion** is the classical Coulomb repulsion between the two protons. The nuclei are
  treated as fixed (the *Born–Oppenheimer approximation*), so this term is just a constant that we
  add back at the end.
- The **Hartree–Fock energy** is our starting point. The true answer must be *lower* (variational principle).

## 2. The electronic Hamiltonian (second quantisation)

The **Hamiltonian** $\hat H$ is the operator for the total energy, and its lowest eigenvalue is the
ground-state energy. In *second quantisation* it is written with operators that **create** ($a_p^\dagger$)
or **destroy** ($a_q$) an electron in spin orbital $p$ or $q$:

$$
\hat H_{\text{el}} \;=\; \sum_{pq} h_{pq}\, a_p^\dagger a_q
\;+\; \tfrac12 \sum_{pqrs} h_{pqrs}\, a_p^\dagger a_q^\dagger a_r a_s
$$

- $h_{pq}$ are **one-electron integrals**: kinetic energy plus attraction to the nuclei.
- $h_{pqrs}$ are **two-electron integrals**: electron–electron repulsion.
- PySCF computed all of these numbers in the previous cell.

The full molecular energy is $E_{\text{total}} = E_{\text{electronic}} + E_{\text{nuclear repulsion}}$.

In [3]:
# The fermionic (electron) Hamiltonian as a sum of creation/annihilation terms.
fermionic_op = problem.hamiltonian.second_q_op()

print(f"Number of fermionic terms: {len(fermionic_op)}")
print("First few terms (operator label -> coefficient in Ha):")
# '+_p' means a_p^dagger (create in spin-orbital p), '-_q' means a_q (destroy in q)
for label, coeff in list(fermionic_op.items())[:6]:
    print(f"  {label:<24} {coeff.real:+.6f}")

Number of fermionic terms: 36
First few terms (operator label -> coefficient in Ha):
  +_0 +_0 -_0 -_0          +0.337855
  +_0 +_1 -_1 -_0          +0.332291
  +_0 +_2 -_2 -_0          +0.337855
  +_0 +_3 -_3 -_0          +0.332291
  +_0 +_0 -_1 -_1          +0.090466
  +_0 +_1 -_0 -_1          +0.090466


## 3. Fermion → qubit mapping (Jordan–Wigner)

Quantum computers apply **Pauli operators** ($I, X, Y, Z$) to qubits, not fermionic operators, so the
Hamiltonian has to be translated.

**The Jordan–Wigner (JW) mapping** is the most direct translation:
- **One qubit per spin orbital.** $|1\rangle$ = orbital occupied, $|0\rangle$ = empty.
- Electrons are **fermions**: swapping two of them flips the sign of the wavefunction (the root of the
  Pauli exclusion principle). JW keeps track of that sign with a *string of Z operators*:

$$
a_j^\dagger \;=\; \Big(\prod_{k<j} Z_k\Big)\,\frac{X_j - iY_j}{2}
$$

After mapping, $\hat H$ becomes a weighted sum of Pauli strings,
$\hat H = \sum_i c_i P_i$. A quantum computer can measure each $P_i$ directly.

**Qubit layout (Qiskit Nature convention):** qubits 0–1 hold the **spin-up (α)** orbitals σg and σu*,
and qubits 2–3 hold the **spin-down (β)** orbitals. Qiskit prints Pauli strings
**right-to-left**, so the rightmost character acts on qubit 0.

*Why JW?* It is the simplest mapping to explain and verify. Alternatives such as the Parity or
Bravyi–Kitaev mappings can save qubits or shorten Z-strings; we may compare them later.

In [4]:
mapper = JordanWignerMapper()

# Translate the fermionic Hamiltonian into a qubit Hamiltonian (a SparsePauliOp).
qubit_op = mapper.map(fermionic_op)

print(f"Qubits needed        : {qubit_op.num_qubits}")
print(f"Number of Pauli terms: {len(qubit_op)}\n")
print(f"{'Pauli string':>12}   coefficient (Ha)")
# Every coefficient must be real because the Hamiltonian is Hermitian (energies are real numbers).
for pauli, coeff in sorted(zip(qubit_op.paulis.to_labels(), qubit_op.coeffs), key=lambda t: -abs(t[1])):
    print(f"{pauli:>12}   {coeff.real:+.6f}")

assert np.allclose(qubit_op.coeffs.imag, 0), "Hamiltonian coefficients should be real"
print(f"\nHamiltonian is Hermitian: {np.allclose(qubit_op.to_matrix(), qubit_op.to_matrix().conj().T)}")

Qubits needed        : 4
Number of Pauli terms: 15

Pauli string   coefficient (Ha)
        IIII   -0.810548
        IIZI   -0.225753
        ZIII   -0.225753
        ZIZI   +0.174643
        IZII   +0.172184
        IIIZ   +0.172184
        IZIZ   +0.168928
        ZIIZ   +0.166145
        IZZI   +0.166145
        IIZZ   +0.120913
        ZZII   +0.120913
        YYYY   +0.045233
        XXYY   +0.045233
        YYXX   +0.045233
        XXXX   +0.045233

Hamiltonian is Hermitian: True


**Reading the output**
- **15 Pauli terms on 4 qubits.** `IIII` is a constant energy shift.
- Terms built only from **I and Z** are *diagonal*: they just count which orbitals are occupied
  (orbital energies plus Coulomb repulsion between occupied orbitals).
- The **XXYY / YYXX / XXXX / YYYY** terms are the interesting ones. They move **both electrons
  together** from the bonding to the antibonding orbital. This *double excitation* is exactly the
  electron correlation that Hartree–Fock misses, and the reason VQE can beat HF.

## 4. Exact classical baseline (our "answer key")

Before trusting a quantum algorithm we need the **right answer**. H₂ in STO-3G needs only 4 qubits, so
the Hamiltonian is a 2⁴ × 2⁴ = **16 × 16 matrix**, small enough to **diagonalise exactly** on a laptop.

We compute the exact ground-state energy **three independent ways**. If all three agree, we can trust
the reference completely:

| Method | What it does |
|---|---|
| **A. `NumPyMinimumEigensolver`** (Qiskit) | Exact diagonalisation of the qubit Hamiltonian. `GroundStateEigensolver` adds the nuclear repulsion for us. |
| **B. `numpy.linalg.eigvalsh`** | We build the 16×16 matrix ourselves and take its smallest eigenvalue. Checks the Qiskit tooling. |
| **C. PySCF FCI** | *Full Configuration Interaction*: exact answer computed directly from chemistry, with **no qubit mapping**. Checks the mapping itself. |

> **Caveat (honest scaling note):** exact diagonalisation costs grow as 2^(qubits). It is trivial here
> but impossible for large molecules. That cost is the motivation for VQE.

In [5]:
# --- Method A: Qiskit's exact eigensolver -----------------------------------
exact_solver = GroundStateEigensolver(mapper, NumPyMinimumEigensolver())
exact_result = exact_solver.solve(problem)

E_electronic_exact = exact_result.groundenergy          # electronic part only
E_exact = exact_result.total_energies[0].real           # electronic + nuclear repulsion

# --- Method B: brute-force diagonalisation of the 16x16 qubit matrix ----------
eigenvalues = np.linalg.eigvalsh(qubit_op.to_matrix())  # eigvalsh: for Hermitian matrices, sorted ascending
E_numpy = eigenvalues[0] + nuclear_repulsion

# --- Method C: PySCF Full Configuration Interaction (no qubits involved) ------
mol = gto.M(atom=f"H 0 0 0; H 0 0 {BOND_LENGTH}", basis="sto-3g", charge=0, spin=0, unit="Angstrom")
mf = scf.RHF(mol).run(verbose=0)                        # restricted Hartree-Fock
E_fci = fci.FCI(mf).kernel()[0]                         # exact energy within the STO-3G basis

print(f"A. Qiskit NumPyMinimumEigensolver : {E_exact:.10f} Ha")
print(f"B. numpy eigvalsh (16x16 matrix)  : {E_numpy:.10f} Ha")
print(f"C. PySCF FCI                      : {E_fci:.10f} Ha")

# All three must agree to (essentially) machine precision.
assert abs(E_exact - E_numpy) < 1e-8 and abs(E_exact - E_fci) < 1e-8, "Exact methods disagree!"
print("\nAll three exact methods agree (|difference| < 1e-8 Ha).")

A. Qiskit NumPyMinimumEigensolver : -1.1373060358 Ha
B. numpy eigvalsh (16x16 matrix)  : -1.1373060358 Ha
C. PySCF FCI                      : -1.1373060358 Ha

All three exact methods agree (|difference| < 1e-8 Ha).


In [6]:
correlation_energy = E_exact - hf_energy

print(f"Hartree-Fock energy (mean field)  : {hf_energy:.6f} Ha")
print(f"Exact ground-state energy (FCI)   : {E_exact:.6f} Ha")
print(f"Correlation energy (Exact - HF)   : {correlation_energy * 1000:.3f} mHa")
print(f"Chemical-accuracy target          : 1.600 mHa")
print(f"\nThe VQE answer must land within 1.6 mHa of {E_exact:.6f} Ha "
      "to count as chemically accurate.")

Hartree-Fock energy (mean field)  : -1.116999 Ha
Exact ground-state energy (FCI)   : -1.137306 Ha
Correlation energy (Exact - HF)   : -20.307 mHa
Chemical-accuracy target          : 1.600 mHa

The VQE answer must land within 1.6 mHa of -1.137306 Ha to count as chemically accurate.


### Phase 1 summary
| Quantity | Value |
|---|---|
| Qubits (JW mapping) | 4 |
| Pauli terms in $\hat H$ | 15 |
| Hartree–Fock energy | ≈ −1.11700 Ha |
| **Exact ground-state energy** | **≈ −1.13731 Ha** |
| Correlation energy that VQE must recover | ≈ −20.3 mHa |

Key takeaways:
- HF misses about **20 mHa**, more than 12× the chemical-accuracy threshold, so a good VQE must
  capture electron correlation.
- `E_exact` is stored and will be the yardstick for the VQE result in the next phases.

**Next (Phase 2):** start from the HF state |0101⟩ and build a *parameterised ansatz* circuit that can
also reach the doubly excited state, the ingredient HF is missing.